In [2]:
import json
import math
import sqlite3
import cloudscraper
import pandas as pd

scraper = cloudscraper.create_scraper()

In [3]:
metro_location = input("Enter metro location (e.g. houston-tx, atlanta-ga, dallas-tx): ").strip().lower()

records = []
rows_per_page = 100

In [4]:
page = 1
last_page = 999

while page <= last_page:
    url = (
        f'https://rentprogress.com/bin/progress-residential/'
        f'property-search.market-{metro_location}.page-{page}'
        f'.rows-{rows_per_page}.nr-{rows_per_page}.json'
    )
    response = scraper.get(url)
    response.raise_for_status()

    data = response.json()
    total = data['recordsFound']
    last_page = math.ceil(total / rows_per_page)

    for item in data['results']:
        location = item.get('location', {})
        records.append({
            'property_id': item.get('propertyId', ''),
            'street': item.get('street', ''),
            'city': item.get('city', ''),
            'state': item.get('state', ''),
            'zip': item.get('zip', ''),
            'beds': item.get('beds', ''),
            'baths': item.get('baths', ''),
            'sqft': item.get('sqft', ''),
            'year_built': item.get('yearBuilt', ''),
            'current_price': item.get('currentPrice', ''),
            'old_price': item.get('oldPrice', ''),
            'price_drop': item.get('priceDrop', False),
            'date_available': item.get('dateAvailable', ''),
            'property_status': item.get('propertyStatus', ''),
            'banner_status': item.get('bannerStatus', ''),
            'market': item.get('market', ''),
            'smart_home': item.get('smartHome', False),
            'solar_panels': item.get('solarPanels', False),
            'community_name': item.get('communityName', ''),
            'lat': location.get('lat', ''),
            'lng': location.get('lng', ''),
            'link': f"https://rentprogress.com{item.get('pageUrl', '')}",
            'thumbnail': item.get('thumbnailImage', ''),
            'metro_location': metro_location,
        })

    print(f'Page {page}/{last_page} — {len(data["results"])} listings (total: {total})')
    page += 1

print(f'\nTotal records scraped: {len(records)}')

Page 1/3 — 100 listings (total: 225)
Page 2/3 — 100 listings (total: 225)
Page 3/3 — 25 listings (total: 225)

Total records scraped: 225


In [5]:
df = pd.DataFrame(records)

dupes = df.duplicated(subset='property_id', keep='first').sum()
print(f'Duplicate rows: {dupes}')
df = df.drop_duplicates(subset='property_id', keep='first')

df.info()
df.head()

Duplicate rows: 0
<class 'pandas.DataFrame'>
RangeIndex: 225 entries, 0 to 224
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_id      225 non-null    str    
 1   street           225 non-null    str    
 2   city             225 non-null    str    
 3   state            225 non-null    str    
 4   zip              225 non-null    str    
 5   beds             225 non-null    int64  
 6   baths            225 non-null    float64
 7   sqft             225 non-null    int64  
 8   year_built       225 non-null    str    
 9   current_price    225 non-null    int64  
 10  old_price        225 non-null    int64  
 11  price_drop       225 non-null    bool   
 12  date_available   225 non-null    str    
 13  property_status  225 non-null    str    
 14  banner_status    225 non-null    str    
 15  market           225 non-null    str    
 16  smart_home       225 non-null    bool   
 17  solar_pan

,property_id,street,city,state,zip,beds,baths,sqft,year_built,current_price,...,banner_status,market,smart_home,solar_panels,community_name,lat,lng,link,thumbnail,metro_location
0,74567,3931 Echo Mountain Dr,Kingwood,TX,77345,4,3.5,2738,1994,2450,...,Move-In Ready,Houston,True,False,,30.07614,-95.182275,https://rentprogress.com/property-details/3931...,https://photos.rentprogress.com/WebPhotos/Hous...,houston-tx
1,158978,5111 Natural Bridge Dr,Kingwood,TX,77345,3,2.5,2563,1991,2140,...,Move-In Ready,Houston,True,False,,30.076885,-95.190269,https://rentprogress.com/property-details/5111...,https://photos.rentprogress.com/WebPhotos/Hous...,houston-tx
2,122095,12114 English Brook Cir,Humble,TX,77346,3,2.0,1640,2003,1735,...,Move-In Ready,Houston,True,False,,29.982167,-95.195414,https://rentprogress.com/property-details/1211...,https://photos.rentprogress.com/WebPhotos/Hous...,houston-tx
3,663723,1134 Desert Oasis Ln,Rosenberg,TX,77471,4,2.5,2866,2007,2055,...,Move-In Ready,Houston,True,False,,29.526299,-95.818748,https://rentprogress.com/property-details/1134...,https://photos.rentprogress.com/WebPhotos/Hous...,houston-tx
4,133311,2011 Pinecreek Pass Ln,Katy,TX,77449,3,2.0,1670,2003,1970,...,Move-In Ready,Houston,True,False,,29.796354,-95.736242,https://rentprogress.com/property-details/2011...,https://photos.rentprogress.com/WebPhotos/Hous...,houston-tx


In [6]:
df.to_csv('progress_residential.csv', index=False)
print(f'Saved {len(df)} rows to progress_residential.csv')

Saved 225 rows to progress_residential.csv


In [7]:
conn = sqlite3.connect('progress_residential.db')
df.to_sql('listings', conn, if_exists='replace', index=False)
print(f'Wrote {len(df)} rows to progress_residential.db -> listings table')
conn.close()

Wrote 225 rows to progress_residential.db -> listings table


In [10]:
conn = sqlite3.connect('progress_residential.db')



ValueError: Table 'select * from listings' already exists.